In [ ]:
# core
import os
import argparse
import copy
import pickle
import random
# numerical / data
import numpy as np
import pandas as pd

# torch
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import datasets, models, transforms
from torchvision.datasets import STL10, ImageFolder
from torchvision.transforms import ToTensor, functional as TF

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
# plotting
import matplotlib.pyplot as plt
import seaborn as sns
# misc
from collections import Counter
from sklearn.metrics import ConfusionMatrixDisplay

import torch
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
print("device name:", torch.cuda.get_device_name(0))

from ultralytics import YOLO
import wandb
from ultralytics import settings


cuda available: True
device count: 1
device name: Tesla T4


In [ ]:
import wandb
print(wandb.__version__)

# login for weights and biases 
wandb.login(key="")

0.25.0


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jan-matthias (jan-matthias-johns-hopkins-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Data import and correct format for YOLOv8

In [ ]:
import kagglehub, os, shutil

src = kagglehub.dataset_download("huanghanchina/pascal-voc-2012")
dst = "/users/jmatthia/deep_learning/data/pascal_voc_2012"

print("SRC:", src)
print("SRC exists:", os.path.exists(src))

print("DST:", dst)
os.makedirs(dst, exist_ok=True)
print("DST exists:", os.path.exists(dst))

print("Copying...")
shutil.copytree(src, dst, dirs_exist_ok=True)
print("Copied.")

voc = os.path.join(dst, "VOCdevkit", "VOC2012")
print("VOC path:", voc)
print("VOC exists:", os.path.exists(voc))
if os.path.exists(voc):
    print("VOC contains:", sorted(os.listdir(voc))[:10])

In [ ]:
import os

BASE = "/users/jmatthia/deep_learning/data/pascal_voc_2012/"  # change only if your base path is different

print("Scanning:", BASE)
for d in sorted(os.listdir(BASE)):
    p = os.path.join(BASE, d)
    if os.path.isdir(p):
        print("\nChecking:", p)
        print("Contains:", sorted(os.listdir(p))[:10])
        has = all(os.path.isdir(os.path.join(p, x)) for x in ["JPEGImages", "Annotations", "ImageSets"])
        print("→ Valid VOC root:", has)

In [ ]:
VOC_ROOT = "/users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012"
IMG_DIR  = os.path.join(VOC_ROOT, "JPEGImages")
SPLIT_DIR = os.path.join(VOC_ROOT, "ImageSets", "Main")
os.makedirs(SPLIT_DIR, exist_ok=True)

random.seed(42)

# get image ids (no extension)
ids = sorted(os.path.splitext(f)[0]
             for f in os.listdir(IMG_DIR)
             if f.endswith(".jpg"))

random.shuffle(ids)

n = len(ids)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)

train_ids = ids[:n_train]
val_ids   = ids[n_train:n_train+n_val]
test_ids  = ids[n_train+n_val:]

def write_txt(filename, id_list):
    with open(os.path.join(SPLIT_DIR, filename), "w") as f:
        f.write("\n".join(id_list))

write_txt("train.txt", train_ids)
write_txt("val.txt", val_ids)
write_txt("test.txt", test_ids)

print("Total:", n)
print("Train:", len(train_ids))
print("Val:", len(val_ids))
print("Test:", len(test_ids))

In [ ]:
for split in ["train", "val", "test"]:
    path = os.path.join(SPLIT_DIR, f"{split}.txt")
    with open(path) as f:
        first = f.readline().strip()
    img_ok = os.path.exists(os.path.join(VOC_ROOT, "JPEGImages", first + ".jpg"))
    ann_ok = os.path.exists(os.path.join(VOC_ROOT, "Annotations", first + ".xml"))
    print(split, "example:", first, "img:", img_ok, "xml:", ann_ok, VOC_ROOT)

In [ ]:
import glob, os

root = "/users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_split"
p = glob.glob(os.path.join(root, "labels", "train", "*.txt"))[0]
print("sample:", p)
with open(p, "r") as f:
    for i in range(10):
        line = f.readline()
        print(repr(line))

In [ ]:
import os

VOC_SPLIT = "/users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_split"
os.makedirs(VOC_SPLIT, exist_ok=True)

yaml_path = os.path.join(VOC_SPLIT, "voc.yaml")

yaml_text = f"""path: {VOC_SPLIT}
train: images/train
val: images/val
test: images/test

names:
  0: aeroplane
  1: bicycle
  2: bird
  3: boat
  4: bottle
  5: bus
  6: car
  7: cat
  8: chair
  9: cow
  10: diningtable
  11: dog
  12: horse
  13: motorbike
  14: person
  15: pottedplant
  16: sheep
  17: sofa
  18: train
  19: tvmonitor
"""

with open(yaml_path, "w") as f:
    f.write(yaml_text)

print("Wrote:", yaml_path)
print("Exists:", os.path.exists(yaml_path))

# YOLOv8 training + logging with weights and biases 

In [ ]:
import os

# W&B config (no wandb.init, no wandb.login)
os.environ["WANDB_PROJECT"] = "hw3-yolov8"
os.environ["WANDB_MODE"] = "online"
os.environ.pop("WANDB_DISABLED", None)

from ultralytics import YOLO, settings
settings.update({"wandb": True, "comet": False})

DATA_YAML = "/users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_split/voc.yaml"

model = YOLO("yolov8m.pt")
model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    project="/content/drive/MyDrive/hw3-yolov8",
    name="yolov8m_wandb",
    pretrained=False,
)

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/users/jmatthia/deep_learning/data/pascal_voc_2012/VOC2012/VOC_split/voc.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8m_wandb, nbs=64, nms=False, opset=None, optimize=False, optimize

Overriding model.yaml nc=80 with nc=20

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytic

# Test

Running the model on our test set

In [ ]:
run_dir = sorted(glob.glob("/content/drive/MyDrive/hw3-yolov8/yolov8m_wandb*"))[-1]
best = run_dir + "/weights/best.pt"

model = YOLO(best)
metrics = model.val(data=DATA_YAML, split="test", imgsz=640, device=0)

print("mAP50-95:", metrics.box.map)     # main mAP
print("mAP50:", metrics.box.map50)
print("mAP75:", metrics.box.map75)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Breakdown by each label

In [ ]:
names = model.names
# metrics.box.maps is commonly per-class mAP50-95 array
if hasattr(metrics.box, "maps"):
    for i, ap in enumerate(metrics.box.maps):
        print(f"{names[i]:<15s} AP50-95={ap:.3f}")